In [ ]:
# ==========================================
# Jupyter 防缓存魔法指令 (放在第一个 Cell 最顶部)
%load_ext autoreload
%autoreload 2
# ==========================================

import sys
sys.path.append('../src')
import math
import numpy as np

# 导入四大核心组件
from orekit_generator import OrekitOrbitGenerator
from geodesy_engine import WGS84GeodesyEngine
from attitude_control import DynamicAttitudeController
from sensor_optics import LimbOpticsSimulator

# 导入底层 Java 3D 向量库，用于计算中心视线
from org.hipparchus.geometry.euclidean.threed import Vector3D # type: ignore

print("================ 临边大气探测：WGS84 高保真全链路系统启动 ================\n")

# ---------------------------------------------------------
# [Phase 1] 系统总装与初始化
# ---------------------------------------------------------
print("🛰️ [1/4] 正在启动 ESA Orekit 动力学内核...")
orbit_sys = OrekitOrbitGenerator(
    prop_model='HPOP',  # HPOP 高精度数值积分
    a=(6378137.0 + 600000.0), e=0.001, i=math.radians(97.79), 
    raan=0.0, arg_pe=math.radians(90), m0=0.0,
    gravity_degree=6, gravity_order=6
)

print("🌍 [2/4] 正在加载 WGS84 高精度大地测量引擎...")
geodesy_sys = WGS84GeodesyEngine()

print("⚙️ [3/4] 正在装配载荷 (姿态控制器 & 光学镜头)...")
# 相机基准下俯 68度安装
attitude_sys = DynamicAttitudeController(mount_pitch_deg=68.0, drift_rate_arcsec_s=0.05, jitter_3sigma_arcsec=1.5)
# 焦距 850mm，靶面 32.5mm
optics_sys = LimbOpticsSimulator(focal_length_mm=850.0, sensor_size_mm=32.5)

# ---------------------------------------------------------
# [Phase 2] 生成基准星历
# ---------------------------------------------------------
print("⏳ 正在推演基准轨道数据...")
# 推演 3 分钟，步长 10 秒
df_ephem = orbit_sys.generate_ephemeris_dataframe(duration_sec=180, step_sec=10)

# ---------------------------------------------------------
# [Phase 3] 全链路 3D 飞行推演 (Time-Marching Loop)
# ---------------------------------------------------------
print("🚀 [4/4] 飞行推演开始！\n")

# 客户指令：本次任务死死盯住 105 km 大气层
TARGET_ALT_M = 105000.0  

# 打印极具工程感的终端表头
print(f"{'T(s)':<4} | {'卫星状态 (纬度, 经度, 绝对海拔)':<32} | {'切点状态 (纬度, 经度, 靶心高度)':<32} | {'镜子补偿':<8} | {'实际探测覆盖(km)':<15}")
print("-" * 105)

for index, row in df_ephem.iterrows():
    t = row['time_sec']
    date = orbit_sys.initial_date.shiftedBy(float(t))
    x, y, z = row['x'], row['y'], row['z']
    vx, vy, vz = row['vx'], row['vy'], row['vz']
    
    # 1. 解算卫星真实的 WGS84 经纬度高程
    sat_lat, sat_lon, sat_alt_m = geodesy_sys.get_sat_lla(x, y, z, date)
    
    # 2. 姿态控制：感知局部地球半径，吐出补偿镜指令和绝对视线角
    mirror_offset, jitter, absolute_los = attitude_sys.get_pointing_command(
        x, y, z, TARGET_ALT_M, t, geodesy_sys, date
    )
    
    # 3. 光学模拟：射线求交，计算探测器边缘对应的绝对高度范围
    alt_min_m, alt_max_m = optics_sys.calculate_altitude_range(
        x, y, z, vx, vy, vz, absolute_los, geodesy_sys, date
    )
    
    # 4. 提取靶心切点(探测目标)的绝对经纬度
    # 复用光学系统的基底算法，算出中心视线的 3D 矢量
    pos_ecef = Vector3D(float(x), float(y), float(z))
    vel_ecef = Vector3D(float(vx), float(vy), float(vz))
    x_dir, _, z_dir = optics_sys._build_local_orbital_frame(pos_ecef, vel_ecef)
    los_center = optics_sys._get_los_vector(x_dir, z_dir, absolute_los)
    
    # 算靶心切点在地球上的经纬度投影
    tgt_lat, tgt_lon, tgt_alt_m = geodesy_sys.get_limb_tangent_lla(
        x, y, z, los_center.getX(), los_center.getY(), los_center.getZ(), date
    )
    
    # 格式化输出
    str_sat = f"{sat_lat:+06.2f}°, {sat_lon:+07.2f}°, {sat_alt_m/1000.0:6.2f}km"
    str_tgt = f"{tgt_lat:+06.2f}°, {tgt_lon:+07.2f}°, {tgt_alt_m/1000.0:6.2f}km"
    
    print(f"{t:03.0f}s | {str_sat:<32} | {str_tgt:<32} | {mirror_offset:+06.3f}° | [{alt_max_m/1000.0:6.2f} ~ {alt_min_m/1000.0:6.2f}]")

print("\n🎉 全链路 WGS84 闭环验证成功！可以向客户交付数据了。")